# Project 1 - Employee

In [1]:
# Import Libraries

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



In [2]:
# 1. Load raw Excel dataset (10,000 rows, 9 columns)
raw_file = "Uncleaned_Employee_Dataset_10000.xlsx"
df_clean = pd.read_excel(raw_file)

In [3]:
# 2. Standardize text columns while keeping raw columns intact
for col in ["First_Name", "Last_Name", "City", "Department"]:
  df_clean[f"{col}_Clean"] = (
      df_clean[col].astype("string").str.strip().str.title().fillna("Unknown")
  )



In [4]:
# Optional acronym fix so 'It' -> 'IT' and 'Hr' -> 'HR'
acronym_map = {"It": "IT", "Hr": "HR"}
df_clean["Department_Clean"] = df_clean["Department_Clean"].replace(acronym_map)



In [5]:
# 3. Parse mixed Join_Date formats into a single datetime column
parsed_dates = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns]")
for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y"]:
  mask = parsed_dates.isna() & df_clean["Join_Date"].notna()
  parsed_dates.loc[mask] = pd.to_datetime(
      df_clean.loc[mask, "Join_Date"], format=fmt, errors="coerce"
  )

median_join_date = parsed_dates.dropna().median()
df_clean["Join_Date_Clean"] = parsed_dates.fillna(median_join_date).dt.normalize()



In [6]:
# 4. Clean numeric columns: set impossible/extreme values to NaN, then impute with rounded whole integers
# Age: valid working range 18 to 65
valid_age = df_clean["Age"].where(df_clean["Age"].between(18, 65))
median_age = int(round(valid_age.median()))
df_clean["Age_Clean"] = valid_age.fillna(median_age).round(0).astype(int)



In [7]:
# Salary: valid range 10,000 to 500,000
valid_salary = df_clean["Salary"].where(
    df_clean["Salary"].between(10000, 500000)
)
median_salary = int(round(valid_salary.median()))
df_clean["Salary_Clean"] = valid_salary.fillna(median_salary).round(0).astype(int)



In [8]:
# Performance_Rating: valid scale 1.0 to 5.0
valid_rating = df_clean["Performance_Rating"].where(
    df_clean["Performance_Rating"].between(1.0, 5.0)
)
median_rating = int(round(valid_rating.median()))
df_clean["Performance_Rating_Clean"] = (
    valid_rating.fillna(median_rating).round(0).astype(int)
)



In [9]:
# 5. Flag duplicate Customer_IDs without deleting rows so you can audit them
df_clean["Is_Duplicate_Customer_ID"] = df_clean.duplicated(
    subset=["Customer_ID"], keep="first"
)



In [10]:
# 6. Export cleaned workbook
output_file = "Cleaned_Employee_Dataset_10000.xlsx"
df_clean.to_excel(output_file, index=False)
print("Saved:", output_file, "| Shape:", df_clean.shape)

Saved: Cleaned_Employee_Dataset_10000.xlsx | Shape: (10000, 18)
